# 02 TF-IDF Retrieval

Черновик для реализации и проверки keyword retrieval: term frequency, IDF, TF-IDF, ranking, top-k.

In [1]:
import re

def tokenize(text: str) -> list[str]:
    text = text.lower()
    text = re.sub(r"[^а-яa-z0-9\- ]", " ", text)
    tokens = text.split()
    return tokens

In [2]:
from collections import Counter
def compute_df(tokenized_documents: list[list[str]]) -> dict[str, int]:
    df = Counter()
    for tokens in tokenized_documents:
        uniq_terms = set(tokens)
        for term in uniq_terms:
            df[term] += 1
    return dict(df)

In [3]:
import math
def compute_idf(df: dict[str, int], total_doc: int) -> dict[str, float]:
    idf = {}
    for term in df:
        idf[term] = math.log(total_doc / df[term])
    return idf



In [4]:
def compute_tfidf(tokens: list[str], idf: dict[str, float]) -> dict[str, float]:
    tf = Counter(tokens)
    tfidf = {}
    for term in tf:
        if term in idf:
            tfidf[term] = tf[term] * idf[term]
    return tfidf

In [5]:
def dot_product(vec1: dict[str, float], vec2: dict[str, float]) -> float:
    score = 0.0
    for term, weight in vec1.items():
        score += weight * vec2.get(term, 0.0)
    return score

In [6]:
def retrieve_tfidf(
        query: str,
        documents: list[str],
        top_k: int = 3,
)-> list[tuple[int, float]]:
    tokenized_documents = [tokenize(doc) for doc in documents]
    df = compute_df(tokenized_documents)
    total_doc = len(documents)
    idf = compute_idf(df, total_doc)
    doc_vectors = [compute_tfidf(tokens, idf) for tokens in tokenized_documents]
    query_tokens = tokenize(query)
    query_vector = compute_tfidf(query_tokens, idf)
    scores = []
    for doc_index, doc_vector in enumerate(doc_vectors):
        score = dot_product(query_vector, doc_vector)
        scores.append((doc_index, score))
        scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:top_k]

In [9]:
documents = [
    # 0
    "Градиентный спуск используется для оптимизации параметров модели машинного обучения",

    # 1
    "Метод оптимизации помогает найти минимум функции потерь и улучшить качество модели",

    # 2
    "Нейронная сеть обучается на данных и постепенно изменяет веса между слоями",

    # 3
    "Обучение модели включает подбор параметров на основе ошибки на обучающей выборке",

    # 4
    "Линейная регрессия используется для предсказания числовых значений",

    # 5
    "Классификация помогает отнести объект к одному из заранее известных классов",

    # 6
    "TF-IDF используется для поиска документов по важным словам в коллекции текстов",

    # 7
    "Inverted index хранит список документов, в которых встречается каждое слово",

    # 8
    "Retriever ищет релевантные фрагменты документов по запросу пользователя",

    # 9
    "Vector search ищет похожие по смыслу фрагменты текста с помощью embeddings",

    # 10
    "Embedding превращает текст в числовой вектор для сравнения смысловой близости",

    # 11
    "Cosine similarity показывает насколько направления двух векторов похожи друг на друга",

    # 12
    "RAG использует поиск по документам чтобы передать языковой модели релевантный контекст",

    # 13
    "Если retriever нашел плохой контекст языковая модель может дать неправильный ответ",

    # 14
    "Chunk это небольшой фрагмент документа который передается в модель как часть контекста",

    # 15
    "Keyword search ищет точные совпадения слов в документах",

    # 16
    "Semantic search помогает находить документы даже если запрос и текст написаны разными словами",

    # 17
    "Сотрудник может восстановить доступ к учетной записи через заявку в Service Desk",

    # 18
    "Для разблокировки пользователя необходимо подтвердить личность и сбросить пароль",

    # 19
    "Пароль должен быть достаточно сложным и регулярно обновляться согласно политике безопасности",

    # 20
    "Многофакторная аутентификация снижает риск компрометации учетной записи",

    # 21
    "Система предотвращения утечек данных контролирует передачу конфиденциальной информации",

    # 22
    "DLP решение анализирует содержимое сообщений файлов и сетевого трафика",

    # 23
    "Средства защиты информации должны регулярно обновляться и проверяться администраторами",

    # 24
    "Контроллер домена управляет учетными записями пользователей и политиками безопасности",

    # 25
    "SeDebugPrivilege позволяет процессу отлаживать другие процессы и может быть опасной привилегией",

    # 26
    "Команда secedit позволяет экспортировать параметры локальной политики безопасности Windows",

    # 27
    "TLS используется для защиты соединения между клиентом и сервером",

    # 28
    "Кошка является домашним животным и часто живет рядом с человеком",

    # 29
    "Автомобильный двигатель преобразует энергию топлива в механическое движение"
]

result = retrieve_tfidf("метод оптимизации параметров", documents, top_k=3)

print(result)

[(1, 18.901679520715224), (0, 14.667071783379441), (3, 7.333535891689721)]
